In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv("/content/1EuroCrop_agricultural_logistics_dataset.csv")
df2 = pd.read_csv("/content/Delivery truck trip data.csv")
df3 = pd.read_csv("/content/dynamic_supply_chain_logistics_dataset.csv")
df4 = pd.read_csv("/content/smart_logistics_dataset.csv")

In [ ]:
df_unified =pd.read_csv("/content/unified_cold_chain_logistics_dataset.csv")

In [ ]:
df_unified

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1016: RuntimeWarning: overflow encountered in square
  sqr = _ensure_numeric((avg - values) ** 2)


,Timestamp,Latitude,Longitude,Temperature,Humidity,Fuel_Consumption_Rate,Route_Distance_KM,Delay_Metric,Asset_Utilization,Source_Dataset
0,2021-01-01 00:00:00+00:00,40.375568,-77.014318,5.743997e-01,NaN,5.136512e+00,NaN,4.998009,NaN,DynamicSupplyChain
1,2021-01-01 01:00:00+00:00,33.507818,-117.036902,-9.753493e+00,NaN,5.101512e+00,NaN,0.984929,NaN,DynamicSupplyChain
2,2021-01-01 02:00:00+00:00,30.020640,-75.269224,-6.491034e+00,NaN,5.090803e+00,NaN,4.972665,NaN,DynamicSupplyChain
3,2021-01-01 03:00:00+00:00,36.649223,-70.190529,-1.512755e-01,NaN,8.219558e+00,NaN,3.095064,NaN,DynamicSupplyChain
4,2021-01-01 04:00:00+00:00,30.001279,-70.012195,2.429448e+00,NaN,5.000075e+00,NaN,3.216077,NaN,DynamicSupplyChain
...,...,...,...,...,...,...,...,...,...,...
93245,2024-06-29 20:00:00+00:00,NaN,NaN,1.254483e+01,1.046057e+23,1.150468e+12,5.038663e+201,73.180202,NaN,AgriculturalLogistics
93246,2024-06-29 21:00:00+00:00,NaN,NaN,5.769750e+07,9.327460e+32,4.224318e+11,4.710606e+307,2181.626165,NaN,AgriculturalLogistics
93247,2024-06-29 22:00:00+00:00,NaN,NaN,1.216248e+04,1.691420e+49,1.926252e+09,NaN,15559.880624,NaN,AgriculturalLogistics
93248,2024-06-29 23:00:00+00:00,NaN,NaN,1.313343e+03,1.018104e+40,4.264034e+09,5.911150e+219,25946.182979,NaN,AgriculturalLogistics


In [ ]:
pip install ydata_profiling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.3/399.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2


In [ ]:
import pandas as pd
import numpy as np
# The import below is for illustration of the solution.
# In a local environment, use: from ydata_profiling import ProfileReport

# 1. Load the unified dataset
df = pd.read_csv('unified_cold_chain_logistics_dataset.csv')

# 2. Convert Timestamp column
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', utc=True)

# 3. CRITICAL: Clean up extreme outliers
# We will use a more aggressive capping for columns that showed issues with ydata_profiling.

# Define columns that have known outlier issues (based on the summary)
outlier_cols = ['Temperature', 'Humidity', 'Fuel_Consumption_Rate', 'Route_Distance_KM', 'Delay_Metric', 'Asset_Utilization']

# A stricter cleaning method: Cap values at a lower, more realistic threshold for problematic columns.
for col in outlier_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan) # Convert 'inf' to NaN first

    if col in ['Humidity', 'Fuel_Consumption_Rate', 'Route_Distance_KM', 'Asset_Utilization']:
        # For these highly polluted columns, use a fixed, safe maximum.
        # Based on the data exploration, 1e+9 seems like a reasonable upper bound to remove extreme synthetic values.
        df[col] = df[col].apply(lambda x: np.nan if x > 1e+9 else x)
    elif col == 'Temperature':
         # Cap Temperature at a more realistic maximum, e.g., 100 for cold chain
         df[col] = df[col].apply(lambda x: np.nan if x > 100 else x)
    elif col == 'Delay_Metric':
        # Cap Delay_Metric at a more realistic maximum, e.g., 1000
        df[col] = df[col].apply(lambda x: np.nan if x > 1000 else x)
    else:
        # For other metrics, use a high percentile (99.99th percentile)
        threshold = df[col].quantile(0.9999)
        df[col] = df[col].apply(lambda x: np.nan if x > threshold else x)


print("Dataset cleaned: Extreme outliers replaced with NaN.")

# 4. Generate and save the clean report (Local execution step - moved to a separate cell)
# profile = ProfileReport(
#     df,
#     title="Unified Cold Chain Logistics Report (CLEANED)",
#     html={'style': {'full_width': True}},
#     sort=None,
#     explorative=True
# )
# profile.to_file("unified_cold_chain_logistics_report_cleaned.html")
# print("Interactive Pandas Profiling Report saved to unified_cold_chain_logistics_report_cleaned.html")

# Outputting a new summary of the cleaned data for confirmation
print("\n--- Numerical Summary (POST-CLEANING) ---")
print(df.select_dtypes(include=['float64']).describe().transpose().to_markdown(numalign="left", stralign="left"))

Dataset cleaned: Extreme outliers replaced with NaN.

--- Numerical Summary (POST-CLEANING) ---
|                       | count   | mean        | std         | min         | 25%      | 50%      | 75%         | max         |
|:----------------------|:--------|:------------|:------------|:------------|:---------|:---------|:------------|:------------|
| Latitude              | 38992   | 34.0732     | 13.9746     | -89.7915    | 30.1278  | 33.9187  | 42.955      | 89.8701     |
| Longitude             | 38992   | -62.1139    | 65.5409     | -179.82     | -102.559 | -79.7304 | -70.2989    | 179.924     |
| Temperature           | 42036   | 8.15009     | 23.3377     | -10         | -9.73446 | -1.61669 | 19.8997     | 99.9823     |
| Humidity              | 1445    | 3.56086e+07 | 1.40953e+08 | 2.76213e-11 | 59.6     | 70.3     | 2107.85     | 9.89683e+08 |
| Fuel_Consumption_Rate | 47761   | 5.90813e+07 | 1.67251e+08 | 0.221312    | 5.14789  | 9.51988  | 4.07464e+06 | 9.99533e+08 |
| Route_

In [ ]:
from ydata_profiling import ProfileReport
import numpy as np

# Generate the profiling report
profile = ProfileReport(
    df,
    title="Unified Cold Chain Logistics Report (CLEANED)",
    html={'style': {'full_width': True}},
    sort=None,
    explorative=True,
    # config_vars={"samples": {"head": 0, "tail": 0, "random": 0}} # Limit sample size
)

# Save the report to an HTML file
profile.to_file("unified_cold_chain_logistics_report_cleaned.html")

print("Interactive Pandas Profiling Report saved to unified_cold_chain_logistics_report_cleaned.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 10/10 [00:04<00:00,  2.25it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Interactive Pandas Profiling Report saved to unified_cold_chain_logistics_report_cleaned.html


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# --- 1. Load Raw Data ---
try:
    df_dynamic = pd.read_csv('dynamic_supply_chain_logistics_dataset.csv')
    df_smart = pd.read_csv('smart_logistics_dataset.csv')
    df_delivery = pd.read_csv('Delivery truck trip data.csv')
    df_crop = pd.read_csv('1EuroCrop_agricultural_logistics_dataset.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}. Ensure all four CSV files are in the current directory.")
    exit()

print("Step 1/5: Raw Data Loaded.")

# Standardized column map for unification
COLUMNS_MAP = {
    'Timestamp': 'Timestamp',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
    'Temperature': 'Temperature',
    'Humidity': 'Humidity',
    'Fuel_Consumption_Rate': 'Fuel_Consumption_Rate',
    'Route_Distance_KM': 'Route_Distance_KM',
    'Delay_Metric': 'Delay_Metric',
    'Asset_Utilization': 'Asset_Utilization',
    'Source_Dataset': 'Source_Dataset'
}

# --- 2. Clean, Standardize, and Unify Data ---

# A. Process df_dynamic
df1 = df_dynamic.rename(columns={
    'timestamp': 'Timestamp', 'vehicle_gps_latitude': 'Latitude', 'vehicle_gps_longitude': 'Longitude',
    'iot_temperature': 'Temperature', 'fuel_consumption_rate': 'Fuel_Consumption_Rate',
    'eta_variation_hours': 'Delay_Metric',
}).assign(
    Humidity=np.nan, Route_Distance_KM=np.nan, Asset_Utilization=np.nan, Source_Dataset='DynamicSupplyChain'
)[list(COLUMNS_MAP.keys())]

# B. Process df_smart
df2 = df_smart.copy()
df2['Waiting_Time_Hours'] = df2['Waiting_Time'] / 60
df2 = df2.rename(columns={
    'Timestamp': 'Timestamp', 'Latitude': 'Latitude', 'Longitude': 'Longitude',
    'Temperature': 'Temperature', 'Humidity': 'Humidity', 'Waiting_Time_Hours': 'Delay_Metric',
    'Asset_Utilization': 'Asset_Utilization'
}).assign(
    Fuel_Consumption_Rate=np.nan, Route_Distance_KM=np.nan, Source_Dataset='SmartLogistics'
)[list(COLUMNS_MAP.keys())]

# C. Process df_delivery
df3 = df_delivery.copy()
# Delay_Metric: 1.0 for delay ('R'), 0.0 for ontime ('G'), NaN otherwise.
df3['Delay_Metric'] = df3.apply(lambda row: 1.0 if row['delay'] == 'R' else (0.0 if row['ontime'] == 'G' else np.nan), axis=1)
df3 = df3.rename(columns={
    'trip_start_date': 'Timestamp', 'Curr_lat': 'Latitude', 'Curr_lon': 'Longitude',
    'TRANSPORTATION_DISTANCE_IN_KM': 'Route_Distance_KM',
}).assign(
    Temperature=np.nan, Humidity=np.nan, Fuel_Consumption_Rate=np.nan,
    Asset_Utilization=np.nan, Source_Dataset='DeliveryTruck'
)[list(COLUMNS_MAP.keys())]

# D. Process df_crop (Handling extreme outliers using a conservative cap)
df4 = df_crop.rename(columns={'Unnamed: 0': 'Timestamp'}).copy()
df4_cols_map = {
    'Storage_Temperature': 'Temperature', 'Storage_Humidity': 'Humidity',
    'Fuel_Consumption': 'Fuel_Consumption_Rate', 'Route_Distance': 'Route_Distance_KM',
    'Delivery_Time': 'Delay_Metric', 'Efficiency_Ratio': 'Asset_Utilization',
}

for old_col, new_col in df4_cols_map.items():
    df4[new_col] = pd.to_numeric(df4[old_col].astype(str).str.replace('inf', str(np.nan), regex=False), errors='coerce')
    df4[new_col] = df4[new_col].replace([np.inf, -np.inf], np.nan)

df4 = df4.assign(
    Latitude=np.nan, Longitude=np.nan, Source_Dataset='AgriculturalLogistics'
)[list(COLUMNS_MAP.keys())]

# E. Concatenate & Final Time Conversion
unified_df = pd.concat([df1, df2, df3, df4], ignore_index=True)
unified_df['Timestamp'] = pd.to_datetime(unified_df['Timestamp'], errors='coerce', utc=True)
unified_df.dropna(subset=['Timestamp'], inplace=True)

print("Step 2/5: Unified Dataset Created.")

# --- 3. Robust Data Cleaning (Addressing extreme outliers & standardization) ---

# Define a maximum plausible value for logistics metrics (Cap at 10^6 based on audit)
# This removes the extreme synthetic outliers found in the report.
MAX_CAP = 1000000.0
CRITICAL_FLOAT_COLS = ['Temperature', 'Humidity', 'Fuel_Consumption_Rate', 'Route_Distance_KM', 'Delay_Metric', 'Asset_Utilization']

for col in CRITICAL_FLOAT_COLS:
    # Set values outside a plausible range to NaN
    # The max cap is necessary for the highly polluted columns
    unified_df[col] = np.where(unified_df[col].abs() > MAX_CAP, np.nan, unified_df[col])
    # Ensure dtypes are float for imputation
    unified_df[col] = pd.to_numeric(unified_df[col], errors='coerce')


# Handle Unit Standardization (Assumptions: all distances are KM, all times are Hours)
# No conversion needed if the columns are only used for modeling ratios, but good practice to standardize names.
# (Note: 'Delay_Metric' is already hours or a binary flag, keeping as is for now.)

# Imputation Strategy for Key Features (Simple Median Imputation for Data Foundation)
imputer = SimpleImputer(strategy='median')

# 1. Impute Temperature (Crucial for spoilage model)
unified_df['Temperature'] = imputer.fit_transform(unified_df[['Temperature']])

# 2. Impute Fuel_Consumption_Rate (Crucial for cost model)
unified_df['Fuel_Consumption_Rate'] = imputer.fit_transform(unified_df[['Fuel_Consumption_Rate']])

print("Step 3/5: Data Cleaned and Critical Features Imputed.")

# --- 4. Feature Engineering for KPIs and Pilot Scope ---

# Calculate Fill Rate Proxy (Simplistic for MVP, needs domain knowledge for real capacity)
# For the purpose of the data foundation, we can only create a binary proxy
unified_df['Load_Capacity_Known'] = unified_df['Asset_Utilization'].notna().astype(int)

# Create a time-based feature from the Timestamp for demand prediction/seasonal effects
unified_df['Hour_of_Day'] = unified_df['Timestamp'].dt.hour
unified_df['Day_of_Week'] = unified_df['Timestamp'].dt.dayofweek

print("Step 4/5: KPIs and Features Engineered.")

# --- 5. Prepare Small Pilot Sandbox (Initial Clustering) ---

# Focus on the data that has coordinates for route planning simulation
geo_data = unified_df.dropna(subset=['Latitude', 'Longitude']).copy()

# Define pilot scope: 5 clusters to simulate regional hubs (e.g., Nashik, Delhi, Mumbai, etc.)
N_CLUSTERS = 5
if len(geo_data) >= N_CLUSTERS:
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    geo_data['Geo_Cluster'] = kmeans.fit_predict(geo_data[['Latitude', 'Longitude']])
    print(f"Initial Clustering (K-Means, K={N_CLUSTERS}) performed on {len(geo_data)} records.")
else:
    print("Not enough geo-data records for clustering simulation.")

# Merge the cluster ID back into the unified dataset
geo_cluster_map = geo_data[['Timestamp', 'Geo_Cluster']]
unified_df = unified_df.merge(geo_cluster_map, on='Timestamp', how='left')


# --- 6. Final Save ---
FINAL_OUTPUT_FILE = 'unified_cold_chain_logistics_CLEANED_PIPELINE.csv'
unified_df.to_csv(FINAL_OUTPUT_FILE, index=False)

print(f"\nStep 5/5: Data Foundation Complete.")
print(f"Final CLEANED dataset saved to: {FINAL_OUTPUT_FILE}")
print(f"Total rows in cleaned dataset: {len(unified_df)}")
print("\nProceed to Step 2: Prototype Route-Matching and Backhaul Optimizer.")

Step 1/5: Raw Data Loaded.
Step 2/5: Unified Dataset Created.
Step 3/5: Data Cleaned and Critical Features Imputed.
Step 4/5: KPIs and Features Engineered.
Initial Clustering (K-Means, K=5) performed on 33065 records.

Step 5/5: Data Foundation Complete.
Final CLEANED dataset saved to: unified_cold_chain_logistics_CLEANED_PIPELINE.csv
Total rows in cleaned dataset: 86370

Proceed to Step 2: Prototype Route-Matching and Backhaul Optimizer.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# --- 1. Load Raw Data ---
try:
    df_dynamic = pd.read_csv('dynamic_supply_chain_logistics_dataset.csv')
    df_smart = pd.read_csv('smart_logistics_dataset.csv')
    df_delivery = pd.read_csv('Delivery truck trip data.csv')
    df_crop = pd.read_csv('1EuroCrop_agricultural_logistics_dataset.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}. Ensure all four CSV files are in the current directory.")
    exit()

print("Step 1/6: Raw Data Loaded.")

# Standardized column map for unification
COLUMNS_MAP = {
    'Timestamp': 'Timestamp',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
    'Temperature': 'Temperature',
    'Humidity': 'Humidity',
    'Fuel_Consumption_Rate': 'Fuel_Consumption_Rate',
    'Route_Distance_KM': 'Route_Distance_KM',
    'Delay_Metric': 'Delay_Metric',
    'Asset_Utilization': 'Asset_Utilization',
    'Source_Dataset': 'Source_Dataset'
}

# --- 2. Clean, Standardize, and Unify Data ---
# (Using robust cleaning logic with np.nan and explicit string handling for stability)

# A. Process df_dynamic
df1 = df_dynamic.rename(columns={
    'timestamp': 'Timestamp', 'vehicle_gps_latitude': 'Latitude', 'vehicle_gps_longitude': 'Longitude',
    'iot_temperature': 'Temperature', 'fuel_consumption_rate': 'Fuel_Consumption_Rate',
    'eta_variation_hours': 'Delay_Metric',
}).assign(
    Humidity=np.nan, Route_Distance_KM=np.nan, Asset_Utilization=np.nan, Source_Dataset='DynamicSupplyChain'
)[list(COLUMNS_MAP.keys())]

# B. Process df_smart
df2 = df_smart.copy()
df2['Waiting_Time_Hours'] = df2['Waiting_Time'] / 60
df2 = df2.rename(columns={
    'Timestamp': 'Timestamp', 'Latitude': 'Latitude', 'Longitude': 'Longitude',
    'Temperature': 'Temperature', 'Humidity': 'Humidity', 'Waiting_Time_Hours': 'Delay_Metric',
    'Asset_Utilization': 'Asset_Utilization'
}).assign(
    Fuel_Consumption_Rate=np.nan, Route_Distance_KM=np.nan, Source_Dataset='SmartLogistics'
)[list(COLUMNS_MAP.keys())]

# C. Process df_delivery
df3 = df_delivery.copy()
df3['Delay_Metric'] = df3.apply(lambda row: 1.0 if row['delay'] == 'R' else (0.0 if row['ontime'] == 'G' else np.nan), axis=1)
df3 = df3.rename(columns={
    'trip_start_date': 'Timestamp', 'Curr_lat': 'Latitude', 'Curr_lon': 'Longitude',
    'TRANSPORTATION_DISTANCE_IN_KM': 'Route_Distance_KM',
}).assign(
    Temperature=np.nan, Humidity=np.nan, Fuel_Consumption_Rate=np.nan,
    Asset_Utilization=np.nan, Source_Dataset='DeliveryTruck'
)[list(COLUMNS_MAP.keys())]

# D. Process df_crop (Robust handling of extreme outliers)
df4 = df_crop.rename(columns={'Unnamed: 0': 'Timestamp'}).copy()
df4_cols_map = {
    'Storage_Temperature': 'Temperature', 'Storage_Humidity': 'Humidity',
    'Fuel_Consumption': 'Fuel_Consumption_Rate', 'Route_Distance': 'Route_Distance_KM',
    'Delivery_Time': 'Delay_Metric', 'Efficiency_Ratio': 'Asset_Utilization',
}

for old_col, new_col in df4_cols_map.items():
    # Convert to string, replace 'inf' string, then coerce to numeric
    df4[new_col] = df4[old_col].astype(str).str.replace('inf', str(np.nan), regex=False)
    df4[new_col] = pd.to_numeric(df4[new_col], errors='coerce')
    df4[new_col] = df4[new_col].replace([np.inf, -np.inf], np.nan)

df4 = df4.assign(
    Latitude=np.nan, Longitude=np.nan, Source_Dataset='AgriculturalLogistics'
)[list(COLUMNS_MAP.keys())]

# E. Concatenate & Final Time Conversion
unified_df = pd.concat([df1, df2, df3, df4], ignore_index=True)
unified_df['Timestamp'] = pd.to_datetime(unified_df['Timestamp'], errors='coerce', utc=True)
unified_df.dropna(subset=['Timestamp'], inplace=True)

print("Step 2/6: Unified Dataset Created.")

# --- 3. Robust Data Cleaning & Imputation ---

# 3.1 Outlier Capping (Removing non-physical data points)
MAX_CAP = 1000000.0 # 1 Million is a safe, high cap for all features
CRITICAL_FLOAT_COLS = ['Temperature', 'Humidity', 'Fuel_Consumption_Rate', 'Route_Distance_KM', 'Delay_Metric', 'Asset_Utilization']
for col in CRITICAL_FLOAT_COLS:
    unified_df[col] = np.where(unified_df[col].abs() > MAX_CAP, np.nan, unified_df[col])
    unified_df[col] = pd.to_numeric(unified_df[col], errors='coerce')

# 3.2 Imputation Strategy (Median Imputation for Data Foundation)
imputer = SimpleImputer(strategy='median')
unified_df['Temperature'] = imputer.fit_transform(unified_df[['Temperature']])
unified_df['Fuel_Consumption_Rate'] = imputer.fit_transform(unified_df[['Fuel_Consumption_Rate']])

print("Step 3/6: Data Cleaned and Critical Features Imputed.")

# --- 4. Pilot Sandbox Preparation (Geographical Clustering) ---
# Necessary prerequisite for Density and Backhaul features
geo_data = unified_df.dropna(subset=['Latitude', 'Longitude']).copy()
N_CLUSTERS = 5 # Simulating the 5 primary hubs/corridors
if len(geo_data) >= N_CLUSTERS:
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    geo_data['Geo_Cluster'] = kmeans.fit_predict(geo_data[['Latitude', 'Longitude']])
    # Merge the cluster ID back into the unified dataset
    geo_cluster_map = geo_data[['Timestamp', 'Geo_Cluster']]
    unified_df = unified_df.merge(geo_cluster_map, on='Timestamp', how='left')
else:
    unified_df['Geo_Cluster'] = np.nan
    print("Not enough geo-data records for clustering simulation. Geo_Cluster set to NaN.")

print("Step 4/6: Initial Geographical Clustering Complete.")

# =========================================================================
# --- 5. FEATURE ENGINEERING (New Step) ---
# =========================================================================

# 5.1 Spoilage Risk Score (Temperature Trends)
# Metric: Absolute mean deviation from an ideal cold chain temperature (5C)
IDEAL_TEMP = 5.0
ROLLING_WINDOW = 5 # 5 pings/observations for local trend

unified_df['Temp_Deviation_Score'] = (unified_df['Temperature'] - IDEAL_TEMP).abs()

# Metric: Temperature Variability (Risk of thermal shock/instability)
unified_df['Temp_Rolling_Std'] = unified_df.sort_values('Timestamp') \
                                         .groupby('Geo_Cluster')['Temperature'] \
                                         .transform(lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).std())

# Final Spoilage Risk is a combination of deviation and variability
unified_df['Spoilage_Risk_Score'] = unified_df['Temp_Deviation_Score'] * 0.7 + unified_df['Temp_Rolling_Std'].fillna(0) * 0.3


# 5.2 Shipment Density Proxy (Demand/Traffic in a region)
# Metric: Count of shipments (pings) in the same cluster over the past 7 days.
unified_df = unified_df.set_index('Timestamp').sort_index()

# Resample by week per cluster and roll the count
weekly_density = unified_df.groupby('Geo_Cluster').resample('7D')['Source_Dataset'].count()
weekly_density.name = 'Shipment_Density'
unified_df = unified_df.merge(weekly_density, on=['Geo_Cluster', 'Timestamp'], how='left')
unified_df['Shipment_Density'] = unified_df.groupby('Geo_Cluster')['Shipment_Density'].ffill().fillna(0)


# 5.3 Estimated Backhaul Availability Proxy
# Metric: Flag if the delivery location (Geo_Cluster) is a high-volume hub (Top 3 clusters)
top_hubs = unified_df['Geo_Cluster'].value_counts().nlargest(3).index.tolist()

unified_df['Backhaul_Potential_Flag'] = unified_df['Geo_Cluster'].apply(
    lambda x: 1 if x in top_hubs and pd.notna(x) else 0
)

# Reset index to original structure
unified_df = unified_df.reset_index()

print("Step 5/6: Feature Engineering Complete.")
print("  - Spoilage_Risk_Score: Created based on Temp deviation and variability.")
print("  - Shipment_Density: Created based on 7-day rolling count per cluster.")
print("  - Backhaul_Potential_Flag: Created based on top 3 geographical hubs.")

# --- 6. Final Save ---
FINAL_OUTPUT_FILE = 'unified_cold_chain_logistics_FULL_PIPELINE.csv'
unified_df.to_csv(FINAL_OUTPUT_FILE, index=False)

print(f"\nStep 6/6: Data Pipeline Complete.")
print(f"Final dataset saved to: {FINAL_OUTPUT_FILE}")
print(f"Total rows in final dataset: {len(unified_df)}")

Step 1/6: Raw Data Loaded.
Step 2/6: Unified Dataset Created.
Step 3/6: Data Cleaned and Critical Features Imputed.
Step 4/6: Initial Geographical Clustering Complete.
Step 5/6: Feature Engineering Complete.
  - Spoilage_Risk_Score: Created based on Temp deviation and variability.
  - Shipment_Density: Created based on 7-day rolling count per cluster.
  - Backhaul_Potential_Flag: Created based on top 3 geographical hubs.

Step 6/6: Data Pipeline Complete.
Final dataset saved to: unified_cold_chain_logistics_FULL_PIPELINE.csv
Total rows in final dataset: 86370


In [ ]:
readdf = pd.read_csv('unified_cold_chain_logistics_FULL_PIPELINE.csv')
readdf

,Timestamp,Latitude,Longitude,Temperature,Humidity,Fuel_Consumption_Rate,Route_Distance_KM,Delay_Metric,Asset_Utilization,Source_Dataset,Geo_Cluster,Temp_Deviation_Score,Temp_Rolling_Std,Spoilage_Risk_Score,Shipment_Density,Backhaul_Potential_Flag
0,2018-06-01 00:00:00+00:00,NaN,NaN,2010.925417,NaN,5.925172,NaN,3468.336745,NaN,AgriculturalLogistics,NaN,2005.925417,NaN,1404.147792,0.0,0
1,2018-06-01 01:00:00+00:00,NaN,NaN,136355.031210,NaN,5.925172,NaN,81.403614,NaN,AgriculturalLogistics,NaN,136350.031210,NaN,95445.021847,0.0,0
2,2018-06-01 02:00:00+00:00,NaN,NaN,1729.900510,NaN,5.925172,NaN,344.008165,NaN,AgriculturalLogistics,NaN,1724.900510,NaN,1207.430357,0.0,0
3,2018-06-01 03:00:00+00:00,NaN,NaN,3322.905892,NaN,5.925172,NaN,1027.148001,NaN,AgriculturalLogistics,NaN,3317.905892,NaN,2322.534125,0.0,0
4,2018-06-01 04:00:00+00:00,NaN,NaN,103.635996,NaN,5.925172,NaN,95.009621,NaN,AgriculturalLogistics,NaN,98.635996,NaN,69.045197,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86365,2024-12-30 02:31:31+00:00,44.5289,31.1002,21.000000,69.4,5.925172,NaN,0.450000,99.9,SmartLogistics,2.0,16.000000,2.780827,12.034248,0.0,0
86366,2024-12-30 07:08:49+00:00,-27.8944,104.5170,29.800000,75.7,5.925172,NaN,1.000000,87.6,SmartLogistics,2.0,24.800000,4.192255,18.617676,0.0,0
86367,2024-12-30 08:15:46+00:00,-29.6864,-88.6901,19.000000,55.4,5.925172,NaN,0.233333,94.5,SmartLogistics,4.0,14.000000,3.814053,10.944216,0.0,0
86368,2024-12-30 19:22:58+00:00,78.1285,162.4365,25.400000,61.5,5.925172,NaN,0.833333,83.4,SmartLogistics,2.0,20.400000,3.671920,15.381576,0.0,0
